# AI ATS Resume Scorer & Evaluator
Evaluates resume text or uploaded files (.pdf, .docx, .txt), analyzes keyword matches, computes an **ATS Score (0-100)**, and provides AI-powered improvement suggestions using **LangGraph** and **Llama 3.3 (HuggingFace)**.

In [1]:
import os, re, io
from dotenv import load_dotenv
from typing import Dict, Any, TypedDict
from langgraph.graph import StateGraph, END

load_dotenv()
openai_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN")

if openai_key and not openai_key.startswith("your_"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    print("LLM Loaded (OpenAI):", llm.model_name)
elif hf_token and not hf_token.startswith("your_"):
    from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
    endpoint = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-3.3-70B-Instruct",
        task="conversational",
        huggingfacehub_api_token=hf_token
    )
    llm = ChatHuggingFace(llm=endpoint)
    print("LLM Loaded (HuggingFace):", endpoint.repo_id)
else:
    raise ValueError("Please set HUGGINGFACEHUB_API_TOKEN or OPENAI_API_KEY in .env")


LLM Loaded (HuggingFace): meta-llama/Llama-3.3-70B-Instruct


In [2]:
# Self-contained ATS Evaluation Engine
TECH_KEYWORDS = {
    "python", "java", "javascript", "typescript", "c++", "golang", "sql", "postgresql",
    "mysql", "mongodb", "redis", "docker", "kubernetes", "aws", "azure", "gcp",
    "ci/cd", "git", "fastapi", "django", "react", "node.js", "rest", "microservices",
    "machine learning", "pandas", "numpy", "linux", "agile", "system design"
}
STANDARD_SECTIONS = ["summary", "experience", "education", "skills", "projects"]

def calculate_ats_metrics(resume_text: str, jd_text: str = "") -> Dict[str, Any]:
    text_lower = resume_text.lower()
    words = re.findall(r"\b[a-zA-Z0-9_\-\+\#\.]+\b", text_lower)
    word_count = len(words)
    
    found_sections = [s for s in STANDARD_SECTIONS if s in text_lower]
    matched_tech = [k for k in TECH_KEYWORDS if k in text_lower]
    
    # Section score (max 30), Skill score (max 40), Length score (max 30)
    sec_score = int((len(found_sections) / len(STANDARD_SECTIONS)) * 30)
    skill_score = min(40, int((len(matched_tech) / 6) * 40))
    len_score = 30 if 300 <= word_count <= 1000 else (15 if word_count > 100 else 5)
    
    overall_score = min(100, sec_score + skill_score + len_score)
    return {
        "overall_score": overall_score,
        "word_count": word_count,
        "found_sections": found_sections,
        "matched_skills": matched_tech
    }

class ATSState(TypedDict):
    resume_text: str
    jd_text: str
    metrics: Dict[str, Any]
    critique: str

def calc_node(state: ATSState):
    return {"metrics": calculate_ats_metrics(state["resume_text"], state.get("jd_text", ""))}

def critique_node(state: ATSState):
    prompt = f"""Provide a structured ATS Resume Critique and 3 actionable recommendations:
Computed ATS Score: {state['metrics']['overall_score']}/100
Detected Skills: {', '.join(state['metrics']['matched_skills']) or 'None'}
Present Sections: {', '.join(state['metrics']['found_sections'])}
Word Count: {state['metrics']['word_count']}
Target Role / JD: {state.get('jd_text', 'Not specified')}

Resume Content Snippet:
{state['resume_text'][:1200]}"""
    res = llm.invoke(prompt)
    return {"critique": res.content if hasattr(res, 'content') else str(res)}

# Build LangGraph Pipeline
builder = StateGraph(ATSState)
builder.add_node("calculate", calc_node)
builder.add_node("critique", critique_node)
builder.set_entry_point("calculate")
builder.add_edge("calculate", "critique")
builder.add_edge("critique", END)
ats_app = builder.compile()
print("ATS LangGraph scoring pipeline compiled successfully!")


ATS LangGraph scoring pipeline compiled successfully!


In [3]:
# Execute Demo Evaluation
demo_resume = """
Alex Johnson
Email: alex.johnson@example.com | LinkedIn: linkedin.com/in/alexjohnson

SUMMARY
Senior Full Stack Software Engineer with 5+ years of experience building resilient cloud systems and APIs.

SKILLS
Languages: Python, SQL, JavaScript, TypeScript
Cloud & DevOps: AWS, Docker, Kubernetes, CI/CD, Git, Linux
Frameworks & DBs: FastAPI, React, PostgreSQL, Redis, Microservices

EXPERIENCE
Senior Software Engineer | CloudTech Solutions (2022 - Present)
- Architected microservices with Python FastAPI and AWS Lambda, reducing latency by 42% for 2M daily users.
- Implemented containerization with Docker and Kubernetes, increasing CI/CD deployment frequency by 65%.
- Optimized PostgreSQL queries and Redis caching, saving $14,000 monthly in cloud infrastructure.

EDUCATION
B.S. in Computer Science | State University
"""

demo_jd = "Seeking a Senior Python Engineer with AWS, Docker, Kubernetes, PostgreSQL, and FastAPI experience."

result = ats_app.invoke({"resume_text": demo_resume, "jd_text": demo_jd})
print(f"\n⭐ ATS SCORE: {result['metrics']['overall_score']} / 100")
print(f"- Word Count: {result['metrics']['word_count']}")
print(f"- Matched Skills: {', '.join(result['metrics']['matched_skills'])}")
print(f"- Present Sections: {', '.join(result['metrics']['found_sections'])}")
print("\n" + "="*60)
print("=== AI ATS CRITIQUE & RECOMMENDATIONS ===")
print(result["critique"])



⭐ ATS SCORE: 79 / 100
- Word Count: 105
- Matched Skills: typescript, java, javascript, linux, ci/cd, redis, react, git, fastapi, python, postgresql, microservices, kubernetes, sql, docker, aws
- Present Sections: summary, experience, education, skills

=== AI ATS CRITIQUE & RECOMMENDATIONS ===
**ATS Resume Critique**

The provided resume has a computed ATS score of 79/100, indicating a good foundation but with opportunities for improvement to increase the chances of passing through applicant tracking systems (ATS) and catching the eye of the hiring manager for a Senior Python Engineer position. Here's a structured critique and three actionable recommendations to enhance the resume:

### Strengths:
- **Relevant Skills**: The resume lists a wide range of relevant technical skills, including Python, AWS, Docker, Kubernetes, PostgreSQL, and FastAPI, which are directly mentioned in the target job description.
- **Quantifiable Achievements**: The experience section includes specific, quant